# The reserved period: checks, then one scoring

[Open in Colab](https://colab.research.google.com/github/Gaire-commits/entropy-gate-pipeline/blob/main/notebooks/reserved_evaluation.ipynb)

Two frozen protocols share the period 2024-01-03 to 2026-10-12: `protocols/volatility_v1.yaml` and `protocols/portfolio_v1.yaml`. **It is reserved for these specifications but previously exposed through related research**; `protocols/reserved_period_disclosure.md` lists every prior use.

This notebook runs in steps. Set `STEP` below, then Runtime → Run all. Each step includes the ones before it, and finished files are kept. Results print here; nothing is downloaded. Paste the output into the chat.

| STEP | What it does | Needs | Time |
|---|---|---|---:|
| **1** | The Alpaca checks: SPY and BIL against the recorded returns, official-open coverage, quoted spreads. **Scores nothing.** | Colab Secrets | about 20 minutes |
| **2** | Rebuilds the audit's sessions, repeats the volatility validation gate on the official open. **Uses no PE and nothing after 2024-01-02.** | Google Drive (the 5-minute cache) | about 40 minutes more |
| **3** | **Scores the reserved period, once.** Volatility first, then the portfolio. It cannot be undone. | `CONFIRM` below, typed exactly | about 30 minutes more |

**Don't save this notebook to GitHub from an old copy.** The code it runs is hash-checked by `tests/test_frozen.py`, so a change to a protocol or its code shows up.

In [ ]:
STEP = 1
CONFIRM = ''   # for STEP 3 only: 'score the reserved period'

In [ ]:
import os, sys
IN_COLAB = 'google.colab' in sys.modules
REPO = '/content/entropy-gate-pipeline' if IN_COLAB else os.environ.get('PORTFOLIO_REPO', os.getcwd())
if IN_COLAB:
    if os.path.isdir(f'{REPO}/.git'):
        !cd {REPO} && git pull -q --ff-only
    else:
        !git clone -q https://github.com/Gaire-commits/entropy-gate-pipeline.git {REPO}
    %cd {REPO}
    !git log --oneline -1
    from google.colab import userdata
    try:
        with open('.env', 'w') as f:
            f.write(f"APCA_API_KEY_ID={userdata.get('APCA_API_KEY_ID')}\nAPCA_API_SECRET_KEY={userdata.get('APCA_API_SECRET_KEY')}\n")
        print('keys written to a local .env (never printed, never committed)')
    except Exception as exc:
        print('No Alpaca keys in Colab Secrets:', type(exc).__name__, '- add them from the key icon, then run this cell again.')
else:
    os.chdir(REPO)
for m in [m for m in sys.modules if m == 'src' or m.startswith('src.')]:
    del sys.modules[m]
sys.path[:0] = [REPO, f'{REPO}/scripts']

from pathlib import Path
REF = os.environ.get('RESERVED_REF', '/content/reserved/reference')
AUDIT = os.environ.get('RESERVED_AUDIT', '/content/reserved/audit')
DRIVE = '/content/drive/MyDrive/entropy-gate-pipeline-data/data'
Path(REF).mkdir(parents=True, exist_ok=True)
Path(AUDIT).mkdir(parents=True, exist_ok=True)

## Step 1: the Alpaca checks

In [ ]:
!python scripts/fetch_reference.py --universe universe/sp500.csv --start 2016-01-04 --end 2026-10-13 --out {REF} --parts daily,corporate_actions,meta,auctions 2>&1 | tail -4
!python scripts/official_prices.py --reference {REF}
!python scripts/protocol_checks.py --reference {REF}

## Step 2: the volatility gate on the official open

In [ ]:
if STEP >= 2:
    if IN_COLAB:
        from google.colab import drive
        drive.mount('/content/drive')
    if not Path(f'{AUDIT}/sessions.parquet').exists():
        !python scripts/data_audit_v2.py --sip {DRIVE}/bars_sip --iex {DRIVE}/bars --universe universe/sp500.csv --reference {REF} --out {AUDIT} 2>&1 | tail -2
    !python scripts/vol_forecast.py --stage baseline --audit {AUDIT} --reference {REF} --official {REF}/official_prices.parquet 2>&1 | grep -v Warning
else:
    print('STEP is', STEP, '- skipped')

## Step 3: score the reserved period, once

Before running this: the protocols are final, the checks above passed, and you have decided to spend this period. Record the outcome in `RESULTS.md` whatever it is.

In [ ]:
if STEP >= 3:
    assert CONFIRM == 'score the reserved period', "set CONFIRM exactly to run this step"
    if not Path(f'{REF}/rv5.parquet').exists():
        !python scripts/vol_forecast.py --stage rv --sip {DRIVE}/bars_sip --rv {REF}/rv5.parquet
    print('=' * 30, 'VOLATILITY')
    !python scripts/vol_forecast.py --stage reserved --score-reserved --audit {AUDIT} --reference {REF} --official {REF}/official_prices.parquet --rv {REF}/rv5.parquet 2>&1 | grep -v Warning
    print('=' * 30, 'PORTFOLIO')
    !python scripts/reserved_portfolio.py --reference {REF} --score-reserved 2>&1 | grep -v Warning
else:
    print('STEP is', STEP, '- the reserved period was not scored')